# SP-HSPLM Stage 1 — Per-token Class B/C rerun on the leak-fixed v3 codebase (A100 / H100)

## Context

Pre-registered protocol: **`docs/SP_HSPLM_Stage_1_pre-registered_protocol.md`**.

Stage 1 reruns the v3 paper's negative-result chain (per-token Class B/C augmentations) under the **leak-fixed** SPLM em_ln codebase, at the **P10g schedule** (TinyStories, d=256, L=8, T=512, 16k steps). The original E1–E5 fitting experiments of paper section 15.5 are leak-immune (they fit a probe to GPT-2 trajectories, not the SPLM training loop), but the *trained-end-to-end* analogue — does a per-token non-conservative term close the SPLM-vs-attention gap on TinyStories — has never been measured cleanly.

This notebook trains one Stage 1 cell at a time, selected by the `CELL` constant. The matched **Cell 0 baseline** (`e0_baseline`) is also a Stage 1 cell — it is the leak-fixed SPLM em_ln baseline at the 16k-step P10g schedule, against which all augmented cells are compared.

### The six cells

| Cell | Force class | Form | Locked rank | Force params |
| --- | --- | --- | ---: | ---: |
| `e0_baseline` | none | g_l = 0, em_ln baseline | n/a | 0 |
| `e1_const_skew` | B (gyroscopic) | g_l = (J − Jᵀ) v_l, J in R^{d×d} | full | ~66 k |
| `e2_affine_rank1` | B (gyroscopic) | g_l = (u hᵀ − h uᵀ) v_l, u in R^d | rank-1 | 256 |
| `e3_lowrank_rank2` | B (gyroscopic) | g_l = (U Hᵀ − H Uᵀ) v_l, H = Wh, low-rank | 2 | ~131 k |
| `e4_solenoidal_rank4` | C (position-only solenoidal) | g_l = (U Vᵀ − V Uᵀ) ρ(h), V = Wh, ρ MLP | 4 | ~296 k |
| `e5_lowrank_rank4` | B (gyroscopic) | g_l = (U Vᵀ − V Uᵀ) v_l, V = Wh | 4 | ~263 k |

### Locked configuration (protocol section 4.1)

TinyStories (5 M train tokens, shard 0); d = 256; L = 8; v_hidden = 1024; v_depth = 3; max_len = 1024; block_size = 512; batch = 16; LR = 5e-4 cosine with 800 warmup; AdamW (betas = (0.9, 0.95), wd = 0.01, grad_clip = 1.0); steps = **16 000** (matched to P10g for forward compatibility with Stage 2). Gamma is *learned* (init 1.0; not fixed). TF32 disabled. `cfg.causal_force = True` (mandatory).

### Pre-registered hypotheses (protocol section 2)

- **H1 (negative result reproduces):** every cell ties Cell 0 within ±2 sigma_seed (about 3.6 PPL).
- **H2 (no diverger):** no cell worsens Cell 0 by more than 5 sigma_seed (about 9 PPL).
- **H3 (causal-leak invariant):** every cell's leak-probe floor at steps 1, 8000, 16000 is at most 1e-6.
- **H4 (Jacobian-symmetry signature, soft):** for cells with non-zero g_l at convergence, the velocity-aware Jacobian is asymmetric, with magnitude bounded by the Frobenius norm of the non-conservative coefficient.

### Decision gates (protocol section 6.3)

- **Outcome ALPHA — negative result reproduces:** all cells tie Cell 0; Stage 2 (SP-HSPLM) is justified.
- **Outcome BETA — surprise on E4-fix:** per-token solenoidal beats Cell 0; pause Stage 2, redesign around per-token solenoidal.
- **Outcome GAMMA — surprise on E1/E2/E3/E5:** Class-B per-token beats Cell 0; rewrite paper section 17.3 hybrid programme.
- **Outcome DELTA — non-conservative collapse:** g_l norm goes to zero across cells; confirms v3 reading; Stage 2 justified.
- **Outcome EPSILON — divergence:** a cell crashes; diagnose (analogous to v3 paper E5 with s = 1).

## 0. Environment setup + cell selector

Pick which Stage 1 cell to run via the `CELL` constant. Architecture is locked to the protocol — only the per-token non-conservative force varies.

**Colab bootstrap (automatic).** When run on Google Colab the next cell:

1. mounts your Google Drive at `/content/drive`,
2. shallow-clones the public `dimitarpg13/semsimula` repo into `/content/semsimula`,
3. symlinks the data cache (`notebooks/conservative_arch/data`) to `/content/drive/MyDrive/semsimula_sp_hsplm/data`,
4. routes all training outputs (ckpt, training log, val PPL plot, causal probe history, non-conservative norms) to `/content/drive/MyDrive/semsimula_sp_hsplm/stage1/{cell}/seed{SEED}/`.

When run locally the notebook walks up from the CWD to find the repo root and writes to `non_conservative/results/sp_hsplm/stage1/{cell}/seed{SEED}/`.

In [ ]:
# ===== Pick the cell to run =====
# One of: 'e0_baseline' | 'e1_const_skew' | 'e2_affine_rank1'
#       | 'e3_lowrank_rank2' | 'e4_solenoidal_rank4' | 'e5_lowrank_rank4'
CELL = 'e0_baseline'
SEED = 0

# ===== Source-of-truth repo + GDrive output dir =====
REPO_URL          = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH       = 'main'
COLAB_REPO_PATH   = '/content/semsimula-paper'
GDRIVE_OUT_REL    = 'semsimula_sp_hsplm'

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.exists():
        try:
            repo_data_dir.rmdir()
        except OSError:
            print(f'NOTE: {repo_data_dir} is non-empty; leaving as-is.')
    if not repo_data_dir.exists():
        repo_data_dir.symlink_to(DATA_CACHE, target_is_directory=True)
        print(f'data cache symlink: {repo_data_dir} -> {DATA_CACHE}')

    RESULTS_ROOT = GDRIVE_OUT / 'stage1'
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow')

else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / 'notebooks').exists():
        raise RuntimeError(
            'Could not locate the semsimula repo root from the notebook CWD.'
        )
    RESULTS_ROOT = (
        REPO_ROOT / 'notebooks' / 'conservative_arch' / 'non_conservative'
        / 'results' / 'sp_hsplm' / 'stage1'
    )
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

NC_DIR      = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'non_conservative'
SCALEUP_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
DATA_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch'
EM_DIR      = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'energetic_minima'
SARFM_DIR   = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'sarf_mass_variant'
for p in (str(REPO_ROOT), str(DATA_DIR), str(SCALEUP_DIR), str(NC_DIR),
          str(EM_DIR), str(SARFM_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'\nREPO_ROOT     = {REPO_ROOT}')
print(f'NC_DIR        = {NC_DIR}')
print(f'RESULTS_ROOT  = {RESULTS_ROOT}')
print(f'CELL = {CELL!r}  SEED = {SEED}')

## 1. Disable TF32, set seeds, pick device

Per the protocol section 4.1, TF32 is disabled mandatorily on CUDA. The SPLM forward uses `torch.autograd.grad(create_graph=True)` for the conservative force, and the second-order path is sensitive to TF32's 10-bit mantissa.

In [ ]:
import torch
import numpy as np

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)
    cap = torch.cuda.get_device_capability()
    print(f'CUDA: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  '
          f'mem={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    if cap[0] < 8:
        print(f'  WARNING: compute capability < 8.0; designed for A100 / H100.')
    print(f'  TF32 matmul = {torch.backends.cuda.matmul.allow_tf32}  (must be False)')
    print(f'  TF32 cuDNN  = {torch.backends.cudnn.allow_tf32}  (must be False)')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('MPS device — TF32 toggles are CUDA-only.')
else:
    device = 'cpu'
    print('CPU only — Stage 1 cells take days on CPU; smoke-test mode strongly recommended.')
print(f'\ndevice = {device}')

## 2. Load TinyStories + the bundled logfreq surprisal

Stage 1 uses the same 5 M-token cap as the P10 ladder (one shard, plus the canonical validation shard).

In [ ]:
from data_module import load_tiny_stories, get_batch

MAX_TRAIN_TOKENS = 5_000_000
N_TRAIN_FILES = 1
train_ids, val_ids = load_tiny_stories(
    n_train_files=N_TRAIN_FILES, max_train_tokens=MAX_TRAIN_TOKENS
)
print(f'tokens: train={len(train_ids):,}  val={len(val_ids):,}')

BUNDLED_LOGFREQ = SCALEUP_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ   = RESULTS_ROOT / 'logfreq_surprisal_tinystories.npy'
if BUNDLED_LOGFREQ.exists():
    LOGFREQ_PATH = BUNDLED_LOGFREQ
    print(f'Using bundled TinyStories logfreq surprisal: {LOGFREQ_PATH}')
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_PATH = DRIVE_LOGFREQ
    print(f'Using Drive-cached TinyStories logfreq surprisal: {LOGFREQ_PATH}')
else:
    VOCAB_SIZE = 50257
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_PATH = DRIVE_LOGFREQ
    LOGFREQ_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_PATH, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_PATH}')

## 3. Build config + model for the selected cell

Reuses `train_splm_nonconservative_scaleup.build_config('scaleup', CELL, ...)` so the notebook is bit-exact to the CLI launch command.

In [ ]:
from train_splm_nonconservative_scaleup import build_config
from model_splm_nonconservative import (
    CELLS, ScalarPotentialLMNonConservative,
)

assert CELL in CELLS, f'CELL must be one of {CELLS}; got {CELL!r}'

cfg, train_cfg = build_config(
    mode='scaleup', cell=CELL, logfreq_path=str(LOGFREQ_PATH),
    fixed_gamma=None,
)

print(f'Stage 1 config locked for cell={CELL}:')
print(f'  d={cfg.d}  L={cfg.L}  v_hidden={cfg.v_hidden}  '
      f'max_len={cfg.max_len}  ln_after_step={cfg.ln_after_step}')
print(f'  causal_force={cfg.causal_force}  (must be True)')
print(f'  steps={train_cfg["steps"]}  warmup={train_cfg["warmup_steps"]}  '
      f'eval_iv={train_cfg["eval_interval"]}  '
      f'probe_steps={train_cfg["probe_steps"]}')

torch.manual_seed(SEED)
model = ScalarPotentialLMNonConservative(cfg).to(device)

n_total = sum(p.numel() for p in model.parameters())
n_force = sum(p.numel() for p in model.nonconservative.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
print(f'\nparams: total={n_total:,}  V_theta={n_v_theta:,}  '
      f'nonconservative={n_force:,}')

full_tag = f'splm_nonconservative_{CELL}_scaleup_seed{SEED}'
print(f'\nfull_tag = {full_tag}')

## 4. Causal-leak probe at init (mandatory per protocol section 4.3)

H3 of the protocol requires the leak floor to be at most 1e-6 at every checkpoint. Run the probe at init; bail out if it fails.

In [ ]:
from train_splm_nonconservative_scaleup import causal_leak_probe

probe_records = []

def run_probe(step: int, label: str):
    rec = causal_leak_probe(
        model, val_ids, train_cfg['block_size'], rng, device,
    )
    rec_full = {'step': step, 'label': label, **rec}
    probe_records.append(rec_full)
    delta = rec['max_logit_delta_past']
    status = 'leak-clean' if delta <= 1e-6 else 'LEAK!'
    print(f'[stage1] causal-leak probe @ step {step} ({label}): '
          f'max_logit_delta_past={delta:.2e} [{status}]')
    if delta > 1e-6:
        raise RuntimeError(
            f'Causal-leak invariant violated (H3 fail): max_logit_delta_past='
            f'{delta:.2e} > 1e-6 at step {step}. Aborting training.'
        )

run_probe(1, 'init')
print('init probe OK — no future-position leak.')

## 5. Train

16 000 steps (matched to P10g), batch=16, T=512, AdamW(0.9, 0.95), cosine LR with 800-step warmup. Wall-clock estimate: H100-80GB about 6-9 h per cell on a single GPU.

To dry-run first, override `STEPS = 200, EVAL_INTERVAL = 100, PROBE_STEPS = (1, 100, 200)` in the cell below before launching.

In [ ]:
import math, time, json
import torch.nn as nn

BATCH = train_cfg['batch_size']
BLOCK = train_cfg['block_size']
STEPS = train_cfg['steps']
LR = train_cfg['lr']
WD = train_cfg['weight_decay']
WARMUP = train_cfg['warmup_steps']
GRAD_CLIP = train_cfg['grad_clip']
EVAL_INTERVAL = train_cfg['eval_interval']
EVAL_ITERS = train_cfg['eval_iters']
LOG_INTERVAL = train_cfg['log_interval']
PROBE_STEPS = set(train_cfg['probe_steps'])

def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

# Pre-built batch for nonconservative norm diagnostics.
xb_diag, _ = get_batch(train_ids, BATCH, BLOCK, rng)
x_diag = torch.from_numpy(xb_diag).to(device)

norms_records = []
def record_norms(step: int):
    stats = model.nonconservative_norm_stats(x_diag)
    rec = {'step': step, **stats}
    norms_records.append(rec)
    if stats['ratio']:
        max_r = max(stats['ratio'])
        mean_r = sum(stats['ratio']) / len(stats['ratio'])
        print(f'  nonconservative norms @ step {step}: '
              f'mean(||g||/||f||)={mean_r:.3f}  '
              f'max(||g||/||f||)={max_r:.3f}')
    else:
        print(f'  nonconservative norms @ step {step}: (cell {CELL} has g_l = 0)')

opt = torch.optim.AdamW(
    model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)
model.train()
record_norms(1)  # baseline before training

log = []
t0 = time.time()
for step in range(STEPS):
    if (step + 1) in PROBE_STEPS and step + 1 != 1:
        label = ('mid' if step + 1 < STEPS else 'final')
        run_probe(step + 1, label)
        record_norms(step + 1)

    for g_param in opt.param_groups:
        g_param['lr'] = lr_at(step)

    xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    opt.step()

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        gamma_val = float(model.gamma.item())
        msg = (f'[{CELL}] step {step + 1:>5}/{STEPS}  '
               f'lr={lr_at(step):.2e}  '
               f'train_loss={loss.item():.4f}  '
               f'gamma={gamma_val:.3f}  '
               f'wall={time.time() - t0:.0f}s')
        print(msg)
    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        gamma_val = float(model.gamma.item())
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'gamma={gamma_val:.3f}')
        log.append({'step': step + 1, 'val_loss': val_loss,
                    'val_ppl': val_ppl, 'train_loss': loss.item(),
                    'gamma': gamma_val})
        record_norms(step + 1)

# Final probe always runs at the last step if not already covered.
if STEPS not in PROBE_STEPS:
    run_probe(STEPS, 'final')

elapsed = time.time() - t0
print(f'\n[{CELL}] Training done.  total wall = {elapsed:.0f}s '
      f'({elapsed / 3600:.2f} h)  '
      f'final val_ppl = {log[-1]["val_ppl"]:.2f}')

## 6. Save checkpoint, training log, probe history, non-conservative norms

In [ ]:
RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

ckpt_path = RUN_DIR / f'{full_tag}_ckpt_latest.pt'
log_path = RUN_DIR / f'{full_tag}_training_log.jsonl'
probe_path = RUN_DIR / f'{full_tag}_causal_probe.json'
norms_path = RUN_DIR / f'{full_tag}_nonconservative_norms.json'

import dataclasses

best_row = min(log, key=lambda r: r['val_ppl'])

torch.save(
    {
        'model_state_dict': model.state_dict(),
        'model_cfg': dataclasses.asdict(cfg),
        'variant': 'sarf_mass_ln_nonconservative',
        'experiment': 'SP_HSPLM_Stage_1',
        'cell': CELL,
        'tag': full_tag,
        'seed': SEED,
        'step': STEPS,
        'final_val_ppl': log[-1]['val_ppl'],
        'best_val_ppl': best_row['val_ppl'],
        'best_step': best_row['step'],
        'final_gamma': float(model.gamma.item()),
        'elapsed_sec': elapsed,
        'n_params': n_total,
        'n_force_params': n_force,
    },
    ckpt_path,
)
with open(log_path, 'w') as f:
    for row in log:
        f.write(json.dumps(row) + '\n')
with open(probe_path, 'w') as f:
    json.dump(probe_records, f, indent=2)
with open(norms_path, 'w') as f:
    json.dump(norms_records, f, indent=2)

print(f'wrote ckpt   : {ckpt_path}')
print(f'wrote log    : {log_path}')
print(f'wrote probes : {probe_path}')
print(f'wrote norms  : {norms_path}')
print(f'\nbest val_ppl  = {best_row["val_ppl"]:.2f} @ step {best_row["step"]}')
print(f'final val_ppl = {log[-1]["val_ppl"]:.2f}')

## 7. Plot val PPL trajectory + reference baselines

Reference lines: SPLM em\_ln 4-channel K-EMA leak-free best (14.78), MatchedGPT scale-up (~8). The Cell 0 baseline (this notebook's `e0_baseline` run) is the new **direct comparator** for every other Stage 1 cell.

In [ ]:
import matplotlib.pyplot as plt

steps_log = [r['step'] for r in log]
ppl_log = [r['val_ppl'] for r in log]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(steps_log, ppl_log, marker='o', color='#3a6ea5',
        label=f'{CELL} (this run)')
ax.axhline(14.78, color='#d4820c', linestyle='-.',
           label='SPLM em_ln K-EMA leak-free best (14.78)')
ax.axhline(8.0, color='#b03030', linestyle=':',
           label='matched-attention reference (~8)')
ax.set_xlabel('train step')
ax.set_ylabel('val PPL (log scale)')
ax.set_yscale('log')
ax.set_title(
    f'SP-HSPLM Stage 1 — {CELL} on TinyStories — val PPL trajectory  '
    f'(best = {min(ppl_log):.2f}, final = {ppl_log[-1]:.2f})'
)
ax.legend(loc='upper right')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_val_ppl.png', dpi=120)
plt.show()

## 8. Save training summary (Markdown)

In [ ]:
summary_path = RUN_DIR / f'{full_tag}_summary.md'
final_gamma = float(model.gamma.item())

with open(summary_path, 'w') as f:
    f.write(f'# Training summary — {full_tag}\n\n')
    f.write('- experiment: SP-HSPLM Stage 1 '
            '(per-token Class B/C rerun on leak-fixed v3 codebase)\n')
    f.write('- protocol: docs/SP_HSPLM_Stage_1_pre-registered_protocol.md\n')
    f.write('- model: ScalarPotentialLMNonConservative (em_ln + per-token g_l)\n')
    f.write(f'- cell: {CELL}\n')
    f.write(f'- corpus: TinyStories ({MAX_TRAIN_TOKENS:,} train tokens)\n')
    f.write(f'- params: total {n_total:,}  V_theta {n_v_theta:,}  '
            f'nonconservative {n_force:,}\n')
    f.write(f'- d={cfg.d}  L={cfg.L}  v_hidden={cfg.v_hidden}  '
            f'max_len={cfg.max_len}  ln_after_step=True  '
            f'causal_force={cfg.causal_force}\n')
    f.write(f'- block_size: {train_cfg["block_size"]}  '
            f'batch_size: {train_cfg["batch_size"]}  '
            f'steps: {train_cfg["steps"]}\n')
    f.write(f'- seed: {SEED}\n')
    f.write(f'- elapsed: {elapsed:.0f} s ({elapsed / 3600:.2f} h)\n')
    f.write(f'\n## Results\n\n')
    f.write(f'- Best val PPL: **{best_row["val_ppl"]:.2f}** '
            f'@ step {best_row["step"]}\n')
    f.write(f'- Final val PPL: {log[-1]["val_ppl"]:.2f}\n')
    f.write(f'- Final gamma: {final_gamma:.4f}\n')
    f.write(f'\n## Causal-leak probe history (H3)\n\n')
    f.write('| step | label | max_logit_delta_past | verdict |\n')
    f.write('|---|---|---:|---|\n')
    for rec in probe_records:
        verdict = 'leak-clean' if rec['max_logit_delta_past'] <= 1e-6 else 'LEAK'
        f.write(f'| {rec["step"]} | {rec["label"]} | '
                f'{rec["max_logit_delta_past"]:.2e} | {verdict} |\n')
    f.write(f'\n## Nonconservative norms (final eval interval)\n\n')
    if norms_records:
        last = norms_records[-1]
        f.write(f'(step {last["step"]})\n\n')
        f.write('| layer | ||f|| | ||g|| | ||g||/||f|| |\n')
        f.write('|---:|---:|---:|---:|\n')
        for ell, (fn, gn, r) in enumerate(zip(
            last['f_norms'], last['g_norms'], last['ratio'],
        )):
            f.write(f'| {ell} | {fn:.3f} | {gn:.3f} | {r:.3f} |\n')

print(f'wrote summary: {summary_path}')

## 9. Stage 1 dashboard — collect results across all 6 cells

Auto-collects best/final val PPL and the final-eval non-conservative norm ratio from every Stage 1 cell present under `RESULTS_ROOT/{cell}/seed{SEED}/`. Re-run this notebook for each cell (changing the `CELL` constant at the top); the dashboard becomes cumulative as cells complete.

In [ ]:
stage1_results = {}
for cell_name in CELLS:
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        stage1_results[cell_name] = None
        continue
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        stage1_results[cell_name] = None
        continue
    rows = [json.loads(line) for line in logs[-1].read_text().splitlines()]
    if not rows:
        stage1_results[cell_name] = None
        continue
    final_ppl = rows[-1]['val_ppl']
    best_ppl = min(r['val_ppl'] for r in rows)
    final_gamma_cell = rows[-1].get('gamma', None)
    # Pull the last non-conservative norm record if present.
    norms_files = sorted(cell_dir.glob('*_nonconservative_norms.json'))
    final_ratio = None
    if norms_files:
        try:
            recs = json.loads(norms_files[-1].read_text())
            if recs and recs[-1].get('ratio'):
                final_ratio = max(recs[-1]['ratio'])
        except json.JSONDecodeError:
            pass
    stage1_results[cell_name] = {
        'best': best_ppl, 'final': final_ppl,
        'gamma': final_gamma_cell, 'max_g_over_f': final_ratio,
    }

# Cell 0 baseline = e0_baseline best (else fall back to '?').
e0 = stage1_results.get('e0_baseline')
e0_best = e0['best'] if e0 is not None else None

print(f'{"cell":<22} {"best":>8} {"final":>8} {"gamma":>8} '
      f'{"max||g||/||f||":>14}    Δ vs Cell 0')
print('-' * 84)
for cell_name in CELLS:
    r = stage1_results[cell_name]
    if r is None:
        print(f'{cell_name:<22} {"—":>8} {"—":>8} {"—":>8} {"—":>14}    (not run)')
        continue
    delta_str = (
        f'{r["best"] - e0_best:>+8.2f}' if e0_best is not None
        and cell_name != 'e0_baseline' else '       —'
    )
    gamma_str = f'{r["gamma"]:8.3f}' if r['gamma'] is not None else '       —'
    ratio_str = (
        f'{r["max_g_over_f"]:14.3f}' if r['max_g_over_f'] is not None
        else f'{"—":>14}'
    )
    print(f'{cell_name:<22} {r["best"]:>8.2f} {r["final"]:>8.2f} '
          f'{gamma_str} {ratio_str}    {delta_str}')

print()
print(f'Cell 0 (e0_baseline) is the comparator. sigma_seed reference '
      f'= 1.8 PPL (E1 multi-seed).')
print(f'Tie band: |Δ| < 3.6 PPL = 2 sigma. Surprise: Δ <= -3.6 PPL.')
print(f'Reference: SPLM em_ln K-EMA leak-free best = 14.78 PPL '
      f'(paper v3 §15, R6.h.1, 4k steps).')

## 10. CLI equivalents (for SLURM / nohup launches)

```bash
# Cell 0 — leak-fixed em_ln baseline at the 16k-step P10g schedule
python notebooks/conservative_arch/non_conservative/train_splm_nonconservative_scaleup.py \
    --mode scaleup --cell e0_baseline --seed 0

# Cell E1 — per-token constant skew (Class B)
python notebooks/conservative_arch/non_conservative/train_splm_nonconservative_scaleup.py \
    --mode scaleup --cell e1_const_skew --seed 0

# Cell E2 — per-token affine-rank-1 skew (Class B)
python notebooks/conservative_arch/non_conservative/train_splm_nonconservative_scaleup.py \
    --mode scaleup --cell e2_affine_rank1 --seed 0

# Cell E3 — per-token low-rank skew, r=2 (Class B)
python notebooks/conservative_arch/non_conservative/train_splm_nonconservative_scaleup.py \
    --mode scaleup --cell e3_lowrank_rank2 --seed 0

# Cell E4 — per-token low-rank solenoidal, r=4 (Class C)
python notebooks/conservative_arch/non_conservative/train_splm_nonconservative_scaleup.py \
    --mode scaleup --cell e4_solenoidal_rank4 --seed 0

# Cell E5 — per-token low-rank skew, r=4 (Class B)
python notebooks/conservative_arch/non_conservative/train_splm_nonconservative_scaleup.py \
    --mode scaleup --cell e5_lowrank_rank4 --seed 0
```

Output dirs:
- Local CLI: `notebooks/conservative_arch/non_conservative/results/sp_hsplm/stage1/`
- This notebook (Colab): `/content/drive/MyDrive/semsimula_sp_hsplm/stage1/{cell}/seed{seed}/`
- This notebook (local): `notebooks/conservative_arch/non_conservative/results/sp_hsplm/stage1/{cell}/seed{seed}/`